In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))
import importlib

from functions import color_palettes
importlib.reload(color_palettes)
from functions.color_palettes import *

from functions import utils
importlib.reload(utils)
from functions.utils import *

from functions import config
importlib.reload(config)
from functions.config import *

import math

set_seed(42)
sns.set_theme(style="whitegrid", context="talk")

c:\Users\aoezek\AppData\Local\miniconda3\envs\circadian\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#folder for saving the plots
base_path = r'C:\\Users\\aoezek\\Desktop\\Thesis\\Thesis_Analysis\\visualisations\\may_visuals\\time_series'

metadata = pd.read_excel(metadata_path)
metadata.rename(columns={"ID": "participant_id"}, inplace=True)
metadata['participant_id'] = metadata['participant_id'].str.replace('AL', 'P', regex=False)
age_gender_map = pd.read_excel(age_gender_map_path)
age_gender_map.rename(columns={"ID": "participant_id"}, inplace=True)

records_df = pd.read_excel(time_series_data_path)

for col in ["post_glucose_inc", "pre_glucose_inc", 
            "post_glucose_abs", "pre_glucose_abs",
            "pre_time_grid", "post_time_grid"]:
    if col in records_df.columns:
        records_df[col] = records_df[col].apply(ast.literal_eval)

post_seq = np.stack(records_df["post_glucose_inc"].values)
pre_seq  = np.stack(records_df["pre_glucose_inc"].values)

In [6]:
post_targets = post_seq.astype(np.float32)
pre_seq = pre_seq.astype(np.float32)
participants = records_df["participant_id"].values

In [ ]:
# same feature subsets as task 1
feature_subset_columns = {
    "Meal":meal_cols,
    "Meal+Meta":meal_cols + meta_cols,
    "Meal+Meta+Glucose": meal_cols + meta_cols + glucose_cols,
    "Meal+Meta+Glucose+Circadian": meal_cols + meta_cols + glucose_cols + circadian_cols,
    "Meal+Circadian": meal_cols + circadian_cols,
    "Meal+Glucose":meal_cols + glucose_cols,
    "Glucose+Meal Time":glucose_plus_time,
}

---
#### Appraches 1&2 : LSTM and TRANSFORMER
---

In [ ]:
set_seed(42)
all_results_df_lstm, preds_lstm, targets_lstm, pids, hyperparams_lstm= run_all_subsets(
    records_df=records_df,
    pre_seq=pre_seq,
    post_targets=post_targets,
    participants=participants,
    metadata=metadata,
    feature_subset_columns=feature_subset_columns,
    n_trials=10
)
with open("hyperparams_lstm.json", "w") as f:
    json.dump(hyperparams_lstm, f, indent=2)

In [ ]:
summary_lstm = print_subset_summary(all_results_df_lstm)

In [ ]:
all_results_df_lstm_best = all_results_df_lstm.loc[all_results_df_lstm.groupby("participant_id")["R2"].idxmax()].reset_index(drop=True).sort_values(['R2'],ascending=False)
all_results_df_lstm_best.merge(age_gender_map, on='participant_id', how='left')
all_results_df_lstm_best_young = all_results_df_lstm_best[all_results_df_lstm_best["age_group_label"] == "Age group 19-32"]
all_results_df_lstm_best_old = all_results_df_lstm_best[all_results_df_lstm_best["age_group_label"] == "Age group 59-76"]


In [ ]:
metrics = ["R2", "MAE", "RMSE"]
age_groups = {
    "young": {"df": all_results_df_lstm_best_young, "title": "Age Group 19-32"},
    "old": {"df": all_results_df_lstm_best_old, "title": "Age Group 59-76"}
}

for age_key, age_info in age_groups.items():
    for metric in metrics:
        df = age_info["df"].copy()
        if metric == "R2":
            plot_df = df[df["R2"] > 0].sort_values(metric, ascending=False)
            ylabel = "Best R²"
        else:
            plot_df = df.sort_values(metric, ascending=True)
            ylabel = f"Best {metric}"
            
        plt.figure(figsize=(12, 6))
        ax = sns.barplot(
            data=plot_df,
            x="participant_id",
            y=metric,
            hue="subset",
            palette=mixed_palette
        )
        plt.xticks(rotation=45, fontsize=12)
        plt.ylabel(ylabel)
        plt.xlabel("Participant")
        plt.title(f"LSTM Model Performance ({metric}) by Subset - {age_info['title']}")
        plt.legend(title="Subset", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=10)
        plt.tight_layout()
        
        filename = f"lstm_{age_key}_{metric.lower()}.png"
        save_path = os.path.join(base_path, filename)
        plt.savefig(save_path, dpi=400)
        
        plt.show()

In [ ]:
#latex tables for results hyperparameters etc for the thesis text
with open("hyperparams_lstm.json") as f:
    hyperparams_lstm = json.load(f)

hyperparams_lstm_df = pd.DataFrame(hyperparams_lstm).T
int_cols = ["hidden", "layers", "batch_size", "epochs"]
hyperparams_lstm_df[int_cols] = hyperparams_lstm_df[int_cols].astype(int)

hyperparams_lstm_df.to_latex(
    "hyperparams_lstm.tex",
    caption="LSTM Hyperparameters",
    label="tab:hyperparams_lstm",
    float_format="%.5f"
)

all_results_df_lstm_best.to_latex(
    "best_lstm_results.tex",
    caption="Best LSTM Results (on the best performed feature subset) per Participant",
    label="tab:best_lstm_results",
    float_format="%.3f"
)

latex_table_lstm = all_results_df_lstm.to_latex(
    "lstm_results_3105.tex",
    index=False,
    caption="LSTM Results",
    label="tab:lstm_results",
    float_format="%.3f"
)
subset_freq=all_results_df_lstm_best['subset'].value_counts()
subset_freq.to_latex(
    "subset_freq_lstm.tex",
    index=True,
    caption="Frequencies of Best Feature Subsets in LSTM Model",
    label="tab:subset_freq",
    float_format="%.3f",
    )

---
#### Transformer
---

In [ ]:
set_seed(42)
all_results_df_transformer, preds_transformer, targets_transformer, pids, hyperparams_transformer= run_all_subsets(
    records_df=records_df,
    pre_seq=pre_seq,
    post_targets=post_targets,
    participants=participants,
    metadata=metadata,
    feature_subset_columns=feature_subset_columns,
    n_trials=10,
    nnmodel='Transformer'
)

In [ ]:
summary_transformer = print_subset_summary(all_results_df_transformer)

In [ ]:
all_results_df_transformer_best = all_results_df_transformer.loc[all_results_df_transformer.groupby("participant_id")["R2"].idxmax()].reset_index(drop=True).sort_values(['R2'],ascending=False)
all_results_df_transformer_best.merge(age_gender_map, on='participant_id', how='left')
all_results_df_transformer_best_young = all_results_df_transformer_best[all_results_df_transformer_best["age_group_label"] == "Age group 19-32"]
all_results_df_transformer_best_old = all_results_df_transformer_best[all_results_df_transformer_best["age_group_label"] == "Age group 59-76"]


In [ ]:
metrics = ["R2", "MAE", "RMSE"]
age_groups = {
    "young": {"df": all_results_df_transformer_best_young, "title": "Age Group 19-32"},
    "old": {"df": all_results_df_transformer_best_old, "title": "Age Group 59-76"}
}

for age_key, age_info in age_groups.items():
    for metric in metrics:
        df = age_info["df"].copy()
        if metric == "R2":
            plot_df = df[df["R2"] > 0].sort_values(metric, ascending=False)
            ylabel = "Best R²"
        else:
            plot_df = df.sort_values(metric, ascending=True)
            ylabel = f"Best {metric}"
            
        plt.figure(figsize=(12, 6))
        ax = sns.barplot(
            data=plot_df,
            x="participant_id",
            y=metric,
            hue="subset",
            palette=mixed_palette
        )
        plt.xticks(rotation=45, fontsize=12)
        plt.ylabel(ylabel)
        plt.xlabel("Participant")
        plt.title(f"Transformer Model Performance ({metric}) by Subset - {age_info['title']}")
        plt.legend(title="Subset", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=10)
        plt.tight_layout()
        
        filename = f"transformer_{age_key}_{metric.lower()}.png"
        save_path = os.path.join(base_path, filename)
        plt.savefig(save_path, dpi=400)
        
        plt.show()

In [ ]:
#latex tables for results hyperparameters etc for the thesis text
with open("hyperparams_transformer.json", "w") as f:
    json.dump(hyperparams_transformer, f, indent=2)

with open("hyperparams_transformer.json") as f:
    hyperparams_transformer = json.load(f)

hyperparams_transformer_df = pd.DataFrame(hyperparams_transformer).T
int_cols = ["d_model", "nhead", "num_layers", "batch_size", "epochs"]
hyperparams_transformer_df[int_cols] = hyperparams_transformer_df[int_cols].astype(int)

hyperparams_transformer_df.to_latex(
    "hyperparams_transformer.tex",
    caption="Transformer Hyperparameters",
    label="tab:hyperparams_transformer",
    float_format="%.5f"
)

all_results_df_transformer_best.to_latex(
    "best_transformer_results.tex",
    caption="Best Transformer Results (on the best performed feature subset) per Participant",
    label="tab:best_transformer_results",
    float_format="%.3f",
    index=False
)

latex_table_transformer = all_results_df_transformer.to_latex(
    "transformer_results_3005.tex",
    index=False,
    caption="Transformer Results",
    label="tab:transformer_results",
    float_format="%.3f"
)

transformer_subset_freq=all_results_df_transformer_best['subset'].value_counts()
transformer_subset_freq.to_latex(
    "subset_freq_transformer.tex",
    index=True,
    caption="Selection Frequencies of Best Feature Subsets in Transformer Model: #out of the total 46 participants in 16 participants Glucose+Meal Time was the best performing subset, followed by Meal+Glucose, Glucose subset has the glucose response summary features extracted from the 10-day CGM recordings of the participants and Meal subset has all the features extracted by LLMs for each meal event.",
    label="tab:subset_freq_transformer",
    float_format="%.3f",
)

---
#### Comparison: LSTM - Transformer
---

In [ ]:
summary_comparison=summary_lstm.merge(summary_transformer, on="subset", suffixes=("_lstm", "_transformer"))
summary_comparison_formatted = pd.DataFrame(index=summary_comparison.index)

for metric in ["R2", "MAE", "RMSE"]:
    for model in ["lstm", "transformer"]:
        mean_col = (f"{metric}_{model}", "mean")
        std_col = (f"{metric}_{model}", "std")
        if mean_col in summary_comparison.columns and std_col in summary_comparison.columns:
            summary_comparison_formatted[f"{metric} {model}"] = [
                f"{mean:.3f} ± {std:.2f}"
                for mean, std in zip(
                    summary_comparison[mean_col],
                    summary_comparison[std_col],
                )
            ]

summary_comparison_formatted

In [ ]:
summary_comparison_formatted.to_latex(
    "summary_comparison.tex", 
    index=True, 
    caption="Comparison Summary of LSTM and Transformer Models", 
    label="tab:time_series_comparison")

---
#### RESIDUAL CORRECTION FUSION on LSTM
---

In [ ]:
#best subset with highes selection frequency 
subset_name = "Glucose+Meal Time"  
#best hyperparameters for the best subset with highes selection frequency 
best_params = {
    "hidden": 128,
    "layers": 2,
    "dropout": 0.217,
    "lr": 0.00060,
    "batch_size": 32,
    "epochs": 30
}
cols = feature_subset_columns[subset_name]
scalars_raw = records_df[cols].values.astype(np.float32)

In [ ]:
fusion_results_df, correctors = fusion_lstm_lopo_flat(
    participants = participants,
    pre_seq = pre_seq,
    scalars_raw= scalars_raw,
    post_targets = post_targets,   
    subset_name = "Glucose+Meal Time",
    best_params = best_params,
    nnmodel = "LSTM"
)

In [ ]:
latex_table_transformer = fusion_results_df.to_latex(
    "fusion_results_1206.tex",
    index=False,
    caption="Fusion model results, residual correction on LSTM model",
    label="tab:lstm_fusion_results",
    float_format="%.3f"
)

In [ ]:
fusion_results_df.merge(age_gender_map, on='participant_id', how='left')

In [ ]:
plot_df = fusion_results_df[fusion_results_df['Alpha_Blend'] > 0].sort_values('Alpha_Blend', ascending=False)
        
plt.figure(figsize=(12, 6))
ax = sns.barplot(
    data=plot_df,
    x="participant_id",
    y='Alpha_Blend',
    alpha=0.8,
    hue='age_group_label',
    palette=[mixed_palette[2], mixed_palette[1]],
)
plt.xticks(rotation=45, fontsize=10)
plt.ylabel('Alpha_Blend')
plt.xlabel("Participant")
plt.title(f"Correction Magnitude Coefficient per Participant (Age-Stratified) - Fusion Model")
plt.legend()
plt.tight_layout()

filename = f"fusion_alpha_byage.png"
save_path = os.path.join(base_path, filename)
plt.savefig(save_path, dpi=400)

plt.show()

In [ ]:
metrics = {"Fusion_R2": "R2", "Fusion_MAE":"MAE", "Fusion_RMSE":"RMSE"}

for column, metric in metrics.items():
    if metric == "R2":
        plot_df = fusion_results_df[fusion_results_df["Fusion_R2"] > 0].sort_values(column, ascending=False)
        ylabel = "R²"
    else:
        plot_df = fusion_results_df.sort_values(column, ascending=True)
        ylabel = f"{metric}"
        
    plt.figure(figsize=(12, 6))
    ax = sns.barplot(
        data=plot_df,
        x="participant_id",
        y=column,
        alpha=0.8,
        hue='age_group',
        palette=[mixed_palette[1], mixed_palette[2]],
    )
    plt.xticks(rotation=45, fontsize=10)
    plt.ylabel(ylabel)
    plt.xlabel("Participant")
    plt.title(f"Fusion Model on LSTM Performance ({metric}) by Participant")
    plt.legend()
    plt.tight_layout()
    
    filename = f"fusion_{metric.lower()}_byage.png"
    save_path = os.path.join(base_path, filename)
    plt.savefig(save_path, dpi=400)
    
    plt.show()

---
##### Comparison LSTM and Residual Corrected LSTM
---

In [ ]:
comparison_fusion_lstm= fusion_results_df[['participant_id', 'Fusion_R2', 'Fusion_RMSE', 'Fusion_MAE']].merge(
    all_results_df_lstm_best[['participant_id', 'R2', 'RMSE', 'MAE']],
    on='participant_id',
    how='inner',
)
comparison_fusion_lstm=comparison_fusion_lstm.rename(columns={'R2': 'LSTM_R2',
                                                              'RMSE': 'LSTM_RMSE',
                                                              'MAE': 'LSTM_MAE'})

In [ ]:
x = np.arange(len(comparison_fusion_lstm))
width = 0.35

fig, ax = plt.subplots(figsize=(16, 10))

ax.bar(x - width/2, comparison_fusion_lstm["LSTM_RMSE"], width, label="Global_LSTM", color=mixed_palette[1])
ax.bar(x + width/2, comparison_fusion_lstm["Fusion_RMSE"], width, label="Fusion", color=mixed_palette[0])

ax.axhline(y=0, color="black", linestyle="-", linewidth=1.2)
ax.set_xticks(x)
ax.set_xticklabels(comparison_fusion_lstm["participant_id"], rotation=45, fontsize=11)
ax.set_ylabel("RMSE")
ax.set_title("LSTM vs Fusion  per Participant (RMSE)")
ax.legend()
plt.tight_layout()
plt.savefig("C:\\Users\\aoezek\\Desktop\\Thesis\\Thesis_Analysis\\visualisations\\may_visuals\\lstm_vs_fusion_1007_rmse.png", dpi=400)
plt.show()